# 01 — Exploratory data analysis (IEEE-CIS)

Goal: understand the fraud rate, missingness, amount and time patterns before modelling. EDA only — production code lives in `ml/src/`.

In [ ]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml.src.data import TARGET, TIME_COL, load_ieee, make_synthetic, time_split

pd.set_option("display.max_columns", 50)

## Load
Set `USE_SYNTHETIC = True` to explore the notebook before the Kaggle download finishes. Use `NROWS` to keep things fast on a laptop.

In [ ]:
USE_SYNTHETIC = False
NROWS = None  # e.g. 200_000

df = make_synthetic(20_000) if USE_SYNTHETIC else load_ieee("../../data/raw", nrows=NROWS)
print(df.shape)
print(f"{df.memory_usage(deep=True).sum() / 1e6:.0f} MB")
df.head()

## 1. Fraud rate
This number decides the metrics: with a few percent fraud, accuracy is meaningless.

In [ ]:
rate = df[TARGET].mean()
print(f"Fraud rate: {rate:.2%}  ({df[TARGET].sum():,} of {len(df):,})")
print(f"Accuracy of always predicting not-fraud: {1 - rate:.2%}")

## 2. Missing values
IEEE-CIS has hundreds of sparse columns. Note which groups are mostly empty.

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
print(f"Columns > 50% missing: {(missing > 0.5).sum()} of {df.shape[1]}")
missing.head(25).plot.barh(figsize=(7, 7), title="Top 25 columns by % missing")
plt.gca().invert_yaxis()
plt.show()

## 3. Transaction amount
Compare fraud vs legit on a log scale.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for label, g in df.groupby(TARGET):
    ax.hist(
        np.log1p(g["TransactionAmt"]),
        bins=60,
        alpha=0.6,
        density=True,
        label="fraud" if label else "legit",
    )
ax.set_xlabel("log(1 + TransactionAmt)")
ax.legend()
ax.set_title("Amount by class")
plt.show()
df.groupby(TARGET)["TransactionAmt"].describe()

## 4. Time patterns
`TransactionDT` is seconds from an unknown start. Derive day and hour.

In [ ]:
df["day"] = df[TIME_COL] // 86_400
df["hour"] = (df[TIME_COL] // 3600) % 24

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df.groupby("hour")[TARGET].mean().plot(ax=axes[0], marker="o", title="Fraud rate by hour")
df.groupby("day").size().plot(ax=axes[1], title="Transactions per day")
plt.tight_layout()
plt.show()

## 5. Categorical breakdowns

In [ ]:
for col in ["ProductCD", "card4", "card6"]:
    if col in df:
        print(
            df.groupby(col)[TARGET]
            .agg(["mean", "size"])
            .sort_values("mean", ascending=False)
            .round(4),
            "\n",
        )

## 6. Time-based split check
Fraud rate should be similar across splits; dates must not overlap.

In [ ]:
split = time_split(df)
split.summary()

## Notes for week 2
- 
- 
- 